# Step 1: Importing the necessary libraries

In [17]:
# For file and system operations
from urllib import request
from pathlib import Path
import os
import yaml
import hashlib
import json
import tempfile
import shutil

# For data manipulation and analysis
import pandas as pd

# For numerical operations
import numpy as np 

# For date and time handling
from datetime import datetime

from dataclasses import dataclass, field
from typing import Iterable, Dict, Optional, List

# For plotting
import matplotlib.pyplot as plt

#  For linear regression
from scipy.stats import linregress

# Attempt to import pycountry for geographic conversions
try:
    import pycountry_convert as pc
except ImportError:
    print("Warning: pycountry_convert not found. Geography mapping will be limited.")
    pc = None

# For suppressing warnings
import warnings
warnings.filterwarnings('ignore')



# Step 1: Load the data

## 1. Geography Mapping Logic
Handles the translation of ISO codes into continents and subregions. It uses a configuration file to define custom subregions (e.g., "Eastern Africa").

In [9]:
# 1️⃣ Set dataset path
iso_dataset_path = Path("dataset/iso_metadata.csv")

# 2️⃣ Download file if it doesn't exist
if not iso_dataset_path.exists():
    print("Downloading iso_metadata.csv...")
    iso_dataset_path.parent.mkdir(parents=True, exist_ok=True)  
    url = "..."
    try:
        response = request.urlopen(url)
        with open(iso_dataset_path, 'w', encoding='utf-8') as f:
            f.write(response.read().decode('utf-8'))
        print("✅ Download complete.")
    except request.URLError as e:
        print(f"❌ Error downloading dataset: {e}")
else:
    print("✅ Dataset already exists. Loading from file.")

# 3️⃣ Load dataset into pandas DataFrame
# Using comma as separator (default for this file)
iso_dataset_items = pd.read_csv(iso_dataset_path)

# 4️⃣ Inspect the first and last 5 rows
print("First 5 rows:")
print(iso_dataset_items.head())

print("\nLast 5 rows:")
iso_dataset_items.tail()

✅ Dataset already exists. Loading from file.
First 5 rows:
                    name  iso
0            Afghanistan  AFG
1  Akrotiri and Dhekelia  XAD
2                  Åland  ALA
3                Albania  ALB
4                Algeria  DZA

Last 5 rows:


,name,iso
242,"Virgin Islands, U.S.",VIR
243,Wallis and Futuna,WLF
244,Yemen,YEM
245,Zambia,ZMB
246,Zimbabwe,ZWE


## Apply the continent mapping

In [6]:
# Continent code → continent name mapping

CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica",
}

# ISO3 codes that don't have standard mappings
SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}

def iso3_to_continent(iso3: str) -> str:
    """
    Converts a 3-letter ISO country code to a continent name.
    Returns "Unknown" if mapping fails.
    """
    if not iso3 or pc is None:
        return "Unknown"

    iso3_clean = iso3.upper().strip()
    if iso3_clean in SPECIAL_ISO3_TO_ISO2:
        return "Unknown"

    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3_clean)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
    except Exception:
        return "Unknown"


class GeographyMapper:
    """
    Handles mapping countries to continents and subregions.
    
    Attributes:
        subregion_map (Dict[str, Dict[str, str]]): Nested dict [continent][ISO3] -> subregion name
    """
    def __init__(self, subregion_config_path: Optional[str | Path] = None):
        self.subregion_map: Dict[str, Dict[str, str]] = {}
        if subregion_config_path:
            self.subregion_map = self._load_subregion_map(Path(subregion_config_path))

    @staticmethod
    def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
        """
        Loads subregion mapping from a YAML file.
        Format: subregions: {continent: {subregion_name: [ISO3, ...]}}
        """
        if not path.exists():
            return {}

        payload = yaml.safe_load(path.read_text()) or {}
        subregions = payload.get("subregions", {})
        mapping: Dict[str, Dict[str, str]] = {}

        for continent, region_dict in subregions.items():
            mapping[continent] = {}
            if not isinstance(region_dict, dict):
                continue
            for subregion_name, iso_list in region_dict.items():
                for iso in iso_list or []:
                    mapping[continent][iso.upper()] = subregion_name
        return mapping

    def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
        """
        Adds 'continent' and 'subregion' columns to a DataFrame based on ISO3 codes.
        """
        out = df.copy()
        # Map continents efficiently
        out["continent"] = out[iso_col].map(iso3_to_continent)

        # Map subregions; defaults to "Unassigned" if no mapping exists
        def map_subregion(row):
            continent = row["continent"]
            iso3 = str(row[iso_col]).upper()
            return self.subregion_map.get(continent, {}).get(iso3, "Unassigned")

        out["subregion"] = out.apply(map_subregion, axis=1)
        return out
    
    # 1️⃣ Create a mapper instance (without subregion config for now)
mapper = GeographyMapper()

# 2️⃣ Add continent and subregion columns
iso_dataset_items_mapped = mapper.add_geography(iso_dataset_items)

# 3️⃣ Inspect the first 5 rows
iso_dataset_items_mapped.head()

,name,iso,continent,subregion
0,Afghanistan,AFG,Asia,Unassigned
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned
2,Åland,ALA,Europe,Unassigned
3,Albania,ALB,Europe,Unassigned
4,Algeria,DZA,Africa,Unassigned


In [18]:
# -----------------------------
# Continent mapping
# -----------------------------
CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica",
}

SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}


def iso3_to_continent(iso3: str) -> str:
    """Convert ISO3 → continent name."""
    if not iso3:
        return "Unknown"

    iso3 = iso3.upper().strip()

    if iso3 in SPECIAL_ISO3_TO_ISO2:
        return "Unknown"

    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
    except Exception:
        return "Unknown"


# -----------------------------
# Geography Mapper
# -----------------------------
class GeographyMapper:

    def __init__(self, subregion_config_path: Optional[str | Path] = None):
        self.subregion_map: Dict[str, Dict[str, str]] = {}

        if subregion_config_path:
            self.subregion_map = self._load_subregion_map(Path(subregion_config_path))

    @staticmethod
    def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
        """Load YAML subregion configuration."""

        if not path.exists():
            return {}

        payload = yaml.safe_load(path.read_text()) or {}
        subregions = payload.get("subregions", {})

        mapping: Dict[str, Dict[str, str]] = {}

        for continent, region_dict in subregions.items():

            mapping[continent] = {}

            if not isinstance(region_dict, dict):
                continue

            for subregion_name, iso_list in region_dict.items():

                for iso in iso_list or []:
                    mapping[continent][iso.upper()] = subregion_name

        return mapping

    # -----------------------------
    # Main method (FIXED)
    # -----------------------------
    def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
        """
        Adds continent and subregion columns.
        Returns only:
        name | iso | continent | subregion
        """

        out = df.copy()

        # continent
        out["continent"] = out[iso_col].map(iso3_to_continent)

        # subregion
        def map_subregion(row):
            continent = row["continent"]
            iso = str(row[iso_col]).upper()

            return self.subregion_map.get(continent, {}).get(iso, "Unassigned")

        out["subregion"] = out.apply(map_subregion, axis=1)

        return out[["name", "iso", "continent", "subregion"]]

# -----------------------------
# Hashing and logging utilities
# -----------------------------

def hash_dataframe(df: pd.DataFrame) -> str:
    """
    Generate a stable hash for a dataframe.
    Much faster than full dataframe comparison.
    """
    df_sorted = df.sort_values(list(df.columns)).reset_index(drop=True)

    # Convert to CSV string (stable representation)
    data_bytes = df_sorted.to_csv(index=False).encode("utf-8")

    return hashlib.sha256(data_bytes).hexdigest()


def log_dataset_version(log_file: Path, info: dict):
    """Append dataset version information to a log file."""
    try:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(info) + "\n")
    except Exception as e:
        print(f"⚠️ Logging failed: {e}")


def safe_write_csv(df: pd.DataFrame, file_path: Path):
    """
    Safely write CSV using a temporary file then move.
    Prevents corruption if write fails.
    """
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
            temp_path = Path(tmp.name)

        df.to_csv(temp_path, index=False)
        shutil.move(temp_path, file_path)

    except Exception as e:
        raise RuntimeError(f"Failed writing CSV: {e}")

# -----------------------------
# Save country geography dataset
# -----------------------------

def save_country_geography(df: pd.DataFrame):

    folder = Path("dataset")
    file_path = folder / "country_geography.csv"
    hash_path = folder / "country_geography.hash"
    log_file = folder / "dataset_versions.log"

    try:

        # -----------------------------
        # 1️⃣ Create folder if needed
        # -----------------------------
        if not folder.exists():
            try:
                folder.mkdir(parents=True, exist_ok=True)
            except Exception as e:
                raise RuntimeError(f"Folder creation failed: {e}")

            df_hash = hash_dataframe(df)

            safe_write_csv(df, file_path)
            hash_path.write_text(df_hash)

            log_dataset_version(
                log_file,
                {
                    "dataset": "country_geography",
                    "timestamp": datetime.utcnow().isoformat(),
                    "version_hash": df_hash,
                    "rows": len(df),
                    "status": "created",
                },
            )

            print('✅ new folder created, "dataset" file "country_geography.csv" saved successfully')
            return

        # -----------------------------
        # 2️⃣ File does not exist
        # -----------------------------
        if not file_path.exists():

            df_hash = hash_dataframe(df)

            safe_write_csv(df, file_path)
            hash_path.write_text(df_hash)

            log_dataset_version(
                log_file,
                {
                    "dataset": "country_geography",
                    "timestamp": datetime.utcnow().isoformat(),
                    "version_hash": df_hash,
                    "rows": len(df),
                    "status": "created",
                },
            )

            print('✅ file "country_geography.csv" saved in dataset successfully')
            return

        # -----------------------------
        # 3️⃣ Compare hashes
        # -----------------------------
        new_hash = hash_dataframe(df)

        if hash_path.exists():
            old_hash = hash_path.read_text().strip()
        else:
            old_hash = None

        if new_hash == old_hash:
            print("✅ file already saved")
            return

        # -----------------------------
        # 4️⃣ Update dataset
        # -----------------------------
        safe_write_csv(df, file_path)
        hash_path.write_text(new_hash)

        log_dataset_version(
            log_file,
            {
                "dataset": "country_geography",
                "timestamp": datetime.utcnow().isoformat(),
                "version_hash": new_hash,
                "rows": len(df),
                "status": "updated",
            },
        )

        print('✅ file updated and saved in dataset successfully')

    except Exception as e:

        print(f"❌ dataset save failed: {e}")

# -----------------------------
# Save the resulting DataFrame to CSV
# -----------------------------
mapper = GeographyMapper("subregions.yaml")

geo_df = mapper.add_geography(iso_dataset_items)

save_country_geography(geo_df)


# -----------------------------
# Run
# -----------------------------
mapper = GeographyMapper("subregions.yaml")

geo_df = mapper.add_geography(iso_dataset_items)

# Check the first 5 rows of the resulting DataFrame
print("First 5 rows of the geographic mapping:")
print(geo_df.head())

# Check the last 5 rows of the resulting DataFrame
print("Last 5 rows of the geographic mapping:")
geo_df.tail()




✅ file "country_geography.csv" saved in dataset successfully
First 5 rows of the geographic mapping:
                    name  iso continent        subregion
0            Afghanistan  AFG      Asia    Southern Asia
1  Akrotiri and Dhekelia  XAD   Unknown       Unassigned
2                  Åland  ALA    Europe  Northern Europe
3                Albania  ALB    Europe  Southern Europe
4                Algeria  DZA    Africa  Northern Africa
Last 5 rows of the geographic mapping:


,name,iso,continent,subregion
242,"Virgin Islands, U.S.",VIR,North America,Caribbean
243,Wallis and Futuna,WLF,Oceania,Polynesia
244,Yemen,YEM,Asia,Western Asia
245,Zambia,ZMB,Africa,Eastern Africa
246,Zimbabwe,ZWE,Africa,Eastern Africa


In [8]:
# -----------------------------
# EDA 
# -----------------------------

# Check the columns of the resulting DataFrame
# print("Columns in the geographic mapping DataFrame:")
# print(geo_df.columns())

# Filter rows with unknown mappings
print("Rows with unknown continent mapping:")
geo_df[geo_df['continent'] == "Unknown"]

print("Rows with unknown subregion mapping:")
geo_df[geo_df['subregion'] == "Unassigned"]

Rows with unknown continent mapping:
Rows with unknown subregion mapping:


,name,iso,continent,subregion
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned
9,Antarctica,ATA,Unknown,Unassigned
28,"Bonaire, Sint Eustatius and Saba",BES,North America,Unassigned
33,British Indian Ocean Territory,IOT,Asia,Unassigned
48,Christmas Island,CXR,Asia,Unassigned
49,Clipperton Island,XCL,Unknown,Unassigned
50,Cocos Islands,CCK,Asia,Unassigned
80,French Southern Territories,ATF,Unknown,Unassigned
116,Kosovo,XKO,Unknown,Unassigned
173,Pitcairn Islands,PCN,Unknown,Unassigned
